In [0]:
from pyspark.sql import functions as F

df = spark.table(
    "workspace.silver.votacoes"
)

display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F

df_gold = (

    df

    .withColumn(
        "sk_votacao",
        F.col("id")
    )

    .withColumn(
        "dt_votacao",
        F.to_date("data")
    )

    .withColumn(
        "dt_hora_registro",
        F.to_timestamp("dataHoraRegistro")
    )

    .withColumn(
        "id_evento",
        F.regexp_extract(
            F.col("uriEvento"),
            r'(\d+)$',
            1
        ).cast("long")
    )

    .withColumn(
        "id_orgao",
        F.regexp_extract(
            F.col("uriOrgao"),
            r'(\d+)$',
            1
        ).cast("long")
    )

    .select(

        "sk_votacao",

        "dt_votacao",

        "dt_hora_registro",

        "descricao",

        "aprovacao",

        "proposicaoObjeto",

        "siglaOrgao",

        "id_evento",

        "id_orgao",

        "uri",

        "uriEvento",

        "uriOrgao",

        "uriProposicaoObjeto",

        "dt_ingestao",

        "dt_processamento"

    )

    .dropDuplicates()

)

display(df_gold)

In [0]:
(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.fato_votacoes"
    )
)

In [0]:
%sql
OPTIMIZE workspace.gold.fato_votacoes
ZORDER BY (
    dt_votacao,
    id_orgao
)

In [0]:
%sql
SELECT *
FROM workspace.gold.fato_votacoes
LIMIT 20